## Chapter 05: Embedding LLMs within Your Applications

### Getting started with LangChain

#### 1. Models and prompts

In [58]:
import environment
import importlib

importlib.reload(environment)

from langchain_deepseek import ChatDeepSeek

In [ ]:
# LangChain model integrations
llm = ChatDeepSeek(
  model="deepseek-chat",
  api_key=config.DEEPSEEK_API_KEY,
  temperature=0,
  max_tokens=None,
  timeout=None,
  max_retries=2,
)

In [7]:
print(
  llm.invoke("Tell me a joke").content
)

Why don't scientists trust atoms?

Because they make up everything!


In [8]:
# Prompt templates
from langchain_core.prompts import PromptTemplate

In [10]:
# Instantiation
template = """
Sentence: {sentence}

Translation in {language}:
"""

prompt = PromptTemplate.from_template(template)

print(
  prompt.format(
    sentence = "The cat is on the table.",
    language = "spanish"
	)
)


Sentence: The cat is on the table.

Translation in spanish:



In [ ]:
# Example selector
from langchain_core.example_selectors import BaseExampleSelector

In [49]:
example = BaseExampleSelector.add_example(
  self=BaseExampleSelector,
  example={
    "prompt": "What is your name?",
    "completion": "I go by Sally.",
	}
)

In [32]:
print(example)

None


#### 2. Data connections

In [35]:
# Document loaders
from langchain_core.document_loaders import BaseLoader
from langchain_community.document_loaders import CSVLoader

In [36]:
loader = CSVLoader(file_path="./data/sample.csv")
data = loader.load()

In [37]:
print(data)

[Document(metadata={'source': './data/sample.csv', 'row': 0}, page_content='Name: John\nAge: 25\nCity: New York'), Document(metadata={'source': './data/sample.csv', 'row': 1}, page_content='Name: Emily\nAge: 28\nCity: Los Angeles'), Document(metadata={'source': './data/sample.csv', 'row': 2}, page_content='Name: Michael\nAge: 22\nCity: Chicago')]


In [39]:
# Document transformers
with open("./data/mountain.txt") as f:
  mountain = f.read()

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [41]:
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=100, 				# number of characters for each chunk
  chunk_overlap=20, 			# number of characters overlapping between a preceding and following chunk
  length_function=len, 		# function used to measure the number of characters
)

texts = text_splitter.create_documents([mountain])

In [42]:
print(texts[0])
print(texts[1])
print(texts[2])

page_content='Amidst the serene landscape, towering mountains stand as majestic guardians of nature's beauty.'
page_content='The crisp mountain air carries whispers of tranquility, while the rustling leaves compose a'
page_content='leaves compose a symphony of wilderness.'


In [121]:
# Text embedding models
from google import genai
# from google.genai import types

# import pandas as pd
import numpy as np
# from sklearn.metrics.pairwise import cosine_similarity

In [117]:
embeddings_model = genai.Client(
  api_key=environment.GEMINI_API_KEY,
)

texts = [
  "Good morning!",
  "Oh, hello!",
  "I want to report an accident.",
  "Sorry to hear that. May I ask your name?",
  "Sure, Mario Rossi.",
]

embeddings = embeddings_model.models.embed_content(
  model="gemini-embedding-001",
  contents=texts,
  # config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")
)

# Create a 3x3 table to show the similarity matrix
# df = pd.DataFrame(
#   cosine_similarity(
#     [e.values for e in embeddings.embeddings]
# 	),
#   index=texts,
#   columns=texts,
# )

In [101]:
# print(df)

In [134]:
print("Embed documents:")
print(
  f"Number of vectors: {np.array(embeddings.embeddings).size}; Dimension of each vector: {np.array(embeddings.embeddings[0].values).ndim}"
)

Embed documents:
Number of vectors: 5; Dimension of each vector: 1


In [133]:
embedded_query = embeddings_model.models.embed_content(
  model="gemini-embedding-001",
  contents="What was the name mentioned in the conversation?"
)

print("Embed query:")
print(
  f"Dimension of the vector: {np.array(embedded_query.embeddings).ndim}"
)
print(
  f"Sample of the first 5 elements of the vector: {np.array(embedded_query.embeddings[0].values)[:5]}"
)

Embed query:
Dimension of the vector: 1
Sample of the first 5 elements of the vector: [-0.03061963  0.00328383  0.01569039 -0.0870063  -0.00871451]
